# Run the 9-balancing-area model

Choose `RUN_MODE`, `UNIT_COMMITMENT`, and a unique `RUN_NAME`, edit the settings relevant to that mode, then run all cells. The checked-in settings start a full-year rolling run; for a first run, use the one-day example in the [README's How to run section](../README.md#how-to-run). Results are stored under `results/run_results/<mode_label>/<run_name>/`; reusing a name resumes or overwrites that named run.

- `daily`: runs the date in `REFERENCE_DATE` (24 hours).
- `monthly`: runs every day in the month containing `REFERENCE_DATE`. To keep unit commitment tractable, each day is solved independently.
- `peak`: runs one continuous 168-hour unit-commitment problem for the week containing the annual peak demand. `REFERENCE_DATE` is ignored.
- `selected_weeks`: runs each date in `WEEK_STARTS` as an independent continuous seven-day problem, with configurable warm-up and look-ahead days.
- `rolling`: runs the full 8,760-hour year as rolling windows. Each solve optimizes seven committed days plus one look-ahead day, carries commitment/dispatch/storage state forward, and checkpoints every completed week. `REFERENCE_DATE` is ignored.

`UNIT_COMMITMENT = "full"` uses binary on/off decisions; `"relaxed"` uses continuous on/off fractions and is faster but approximate; `"none"` removes on/off decisions and minimum stable output from formerly committable generators. In rolling relaxed mode, fractional status, transition history, dispatch, and storage state are carried across each window boundary. Non-full modes save to separate result folders.

Gas and nuclear daily energy are equality constrained in every mode. Results are saved automatically under `results/run_results/`. Rolling mode resumes existing completed checkpoints when `RESUME = True`.

After a rolling run, use `analyze_rolling_year.ipynb` for the standardized full-year analysis, including monthly generation, resource seasonality, and boundary-condition checks.

In [12]:
# ===================== USER SETTINGS =====================
# 1. Period to solve
# daily: one date; monthly: independent days in that date's month; peak: peak-demand week;
# selected_weeks: independent weeks listed below; rolling: full year in 7-day steps.
RUN_MODE = "rolling"
REFERENCE_DATE = "2025-07-11"  # YYYY-MM-DD; used by daily and monthly only
# Only selected_weeks uses the next four settings. Each listed date begins a retained 7-day week.
WEEK_STARTS = [
    "2025-04-07",  # dry / low hydro
    "2025-07-07",  # southwest-monsoon wind and hydro peak
    "2025-10-13",  # monsoon transition
    "2026-02-16",  # dry-season low renewables
]
WARMUP_DAYS = 1             # extra days solved before each selected week; excluded from results
LOOKAHEAD_DAYS = 1          # extra days solved after each selected week; excluded from results
RESERVOIR_INITIAL_SOC_FRACTION = 0.5  # 0 to 1; fallback initial fill when no observed/proxy level exists

# 2. Formulation and solver
# full: binary on/off decisions (MIP); relaxed: continuous on/off fractions (LP, approximate);
# none: no on/off decisions or minimum stable output for formerly committable units.
UNIT_COMMITMENT = "none"  # full, relaxed, or none
SOLVER_NAME = "highs"     # supported solver for this workflow
SOLVER_THREADS = 2          # HiGHS threads per solve; integer from 1 to 8
PARALLEL_WORKERS = 1        # simultaneous independent daily/monthly/selected-week solves; rolling/peak use 1
# For parallel runs, active workers x SOLVER_THREADS must fit within logical CPU count.
DISABLE_PARALLELIZATION = False  # True forces 1 solver thread and 1 worker
MIP_REL_GAP = 0.01         # full mode only; dimensionless target gap (0.01 = 1%)

# 3. Rolling-window limits and saved results
ROLLING_TIME_LIMIT = None  # seconds per rolling solve; None means unlimited; LP still requires optimality
ROLLING_MAX_TIME_LIMIT_MIP_GAP = 0.10  # full rolling mode only: largest accepted gap if time limit is hit (10%)
RUN_NAME = "no_parallelisation_LP_test_22"  # output subfolder; use a new name when comparing parallel settings
RESUME = False              # rolling/selected_weeks: reuse completed checkpoints when available
# =========================================================

In [13]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists():
    ROOT = ROOT.parent
if not (ROOT / "scripts").exists():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks folder.")

sys.path.insert(0, str(ROOT / "scripts"))
import importlib
import highs_solver_options
import run_9ba_rolling
import run_9ba_selection

importlib.reload(highs_solver_options)
importlib.reload(run_9ba_rolling)
importlib.reload(run_9ba_selection)
run_selection = run_9ba_selection.run_selection

result = run_selection(
    RUN_MODE,
    REFERENCE_DATE,
    week_starts=WEEK_STARTS,
    warmup_days=WARMUP_DAYS,
    lookahead_days=LOOKAHEAD_DAYS,
    reservoir_initial_soc_fraction=RESERVOIR_INITIAL_SOC_FRACTION,
    solver_name=SOLVER_NAME,
    solver_threads=1 if DISABLE_PARALLELIZATION else SOLVER_THREADS,
    parallel_workers=1 if DISABLE_PARALLELIZATION else PARALLEL_WORKERS,
    mip_rel_gap=MIP_REL_GAP,
    unit_commitment=UNIT_COMMITMENT,
    run_name=RUN_NAME,
    rolling_time_limit=ROLLING_TIME_LIMIT,
    rolling_max_time_limit_mip_gap=ROLLING_MAX_TIME_LIMIT_MIP_GAP,
    resume=RESUME,
)

Loading C:\Users\b076218\Documents\Filkassen\DEA_GE_github\PyPSA-TamilNadu\model ...


c:\Users\b076218\AppData\Local\anaconda3\envs\pypsa\Lib\site-packages\pypsa\io.py:156: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(fn, index_col=0, encoding=self.encoding, parse_dates=True)
c:\Users\b076218\AppData\Local\anaconda3\envs\pypsa\Lib\site-packages\pypsa\io.py:182: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(
c:\Users\b076218\AppData\Local\anaconda3\envs\pypsa\Lib\site-packages\pypsa\io.py:182: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(
c:\Users\b076218\AppData\Local\anaconda3\envs\pypsa\Lib\site-packages

Loaded network from disk in 0.62 s.
Solving window 1/53: 2025-04-01 to 2025-04-08 (168 committed + 24 look-ahead hours)


INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
 - threads: 2
 - parallel: on
 - solver: simplex
 - simplex_strategy: 3
 - simplex_max_concurrency: 2
 - include_objective_constant: False
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 7/7 [00:00<00:00, 12.77it/s]
INFO:linopy.io: Writing time: 4.24s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 95232 primals, 255815 duals
Objective: 6.47e+09
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-fix-p-ramp_limit_up, Generator-fix-p-ramp_limit_down, Link-fix-p-lower, Link-fix-p-upper, Store-fix-e-lower, Store-fix-e-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state

Window 1/53: 15.2 s wall, 15.3 s CPU (1.01 cores average), 806 MB peak RSS, 29 peak process threads. Metrics: C:\Users\b076218\Documents\Filkassen\DEA_GE_github\PyPSA-TamilNadu\results\run_results\rolling_2025-04-01_2026-03-31_none_uc\no_parallelisation_LP_test_22\window_log.partial.csv
Solving window 2/53: 2025-04-08 to 2025-04-15 (168 committed + 24 look-ahead hours)


...

Window 51/53: 10.3 s wall, 10.4 s CPU (1.01 cores average), 1209 MB peak RSS, 29 peak process threads. Metrics: C:\Users\b076218\Documents\Filkassen\DEA_GE_github\PyPSA-TamilNadu\results\run_results\rolling_2025-04-01_2026-03-31_none_uc\no_parallelisation_LP_test_22\window_log.partial.csv
Solving window 52/53: 2026-03-24 to 2026-03-31 (168 committed + 24 look-ahead hours)


INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
 - threads: 2
 - parallel: on
 - solver: simplex
 - simplex_strategy: 3
 - simplex_max_concurrency: 2
 - include_objective_constant: False
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 7/7 [00:00<00:00, 11.98it/s]
INFO:linopy.io: Writing time: 4.23s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 95232 primals, 255815 duals
Objective: 4.77e+09
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-fix-p-ramp_limit_up, Generator-fix-p-ramp_limit_down, Link-fix-p-lower, Link-fix-p-upper, Store-fix-e-lower, Store-fix-e-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state

Window 52/53: 15.5 s wall, 15.8 s CPU (1.02 cores average), 1209 MB peak RSS, 29 peak process threads. Metrics: C:\Users\b076218\Documents\Filkassen\DEA_GE_github\PyPSA-TamilNadu\results\run_results\rolling_2025-04-01_2026-03-31_none_uc\no_parallelisation_LP_test_22\window_log.partial.csv
Solving window 53/53: 2026-03-31 to 2026-03-31 (24 committed + 0 look-ahead hours)


INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
 - threads: 2
 - parallel: on
 - solver: simplex
 - simplex_strategy: 3
 - simplex_max_concurrency: 2
 - include_objective_constant: False
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 7/7 [00:00<00:00, 124.98it/s]
INFO:linopy.io: Writing time: 0.56s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 11904 primals, 31689 duals
Objective: 6.52e+08
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-fix-p-ramp_limit_up, Generator-fix-p-ramp_limit_down, Link-fix-p-lower, Link-fix-p-upper, Store-fix-e-lower, Store-fix-e-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state

Window 53/53: 3.8 s wall, 3.8 s CPU (1.00 cores average), 721 MB peak RSS, 28 peak process threads. Metrics: C:\Users\b076218\Documents\Filkassen\DEA_GE_github\PyPSA-TamilNadu\results\run_results\rolling_2025-04-01_2026-03-31_none_uc\no_parallelisation_LP_test_22\window_log.partial.csv
Finished. Results: C:\Users\b076218\Documents\Filkassen\DEA_GE_github\PyPSA-TamilNadu\results\run_results\rolling_2025-04-01_2026-03-31_none_uc\no_parallelisation_LP_test_22


In [ ]:
from IPython.display import Image, display
import pandas as pd

print(f"Mode: {result['mode']}")
print(f"Period: {result['dates'][0].date()} to {result['dates'][-1].date()}")
print(f"Saved to: {result['output_dir']}")
if result.get('week_dirs'):
    print('Per-week results:')
    for path in result['week_dirs']:
        print(f'  {path}')
display(result["summary"].round(3))
display(pd.Series(result["validation"], name="value").to_frame())
display(Image(filename=str(result["plot_path"])))